# Skripsi Dhio — Klasifikasi Flora Endemik Kalimantan

**Perubahan dari V9 :**
- menghilangkan early stopping
- menghilangkan learning rate schedule dan weight decay

Struktur Drive:
```
MyDrive/
└─ Database Skripsi Dhio/
   ├─ Dataset Mentah/
   │  ├─ Spesies_A/  (100+ citra mentah)
   │  ├─ Spesies_B/
   │  └─ ...
   └─ Dataset/  
```

## 0) Runtime & GPU

In [ ]:
# Pastikan runtime GPU aktif (Runtime → Change runtime type → GPU)
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

TF version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 1) Install KerasCV (untuk RandAugment)

In [ ]:
!pip install keras-cv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 650.7/650.7 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 950.8/950.8 kB 65.3 MB/s eta 0:00:00


## 2) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3) Konfigurasi path (Sesuai Drive Dhio)


In [ ]:
# === KONFIGURASI ===
PROJECT_DIR = "/content/drive/MyDrive/Database_Skripsi_Dhio"
DATASET_RAW = "/content/drive/MyDrive/Database_Skripsi_Dhio/Dataset_Citra"
DATASET     = "/content/drive/MyDrive/Database_Skripsi_Dhio/Dataset/Random"

DO_SPLIT = True

import os
os.makedirs(PROJECT_DIR, exist_ok=True)
%cd $PROJECT_DIR
print("Project dir:", PROJECT_DIR)
print("Raw dataset:", DATASET_RAW)
print("Split out   :", DATASET)

/content/drive/MyDrive/Database_Skripsi_Dhio
Project dir: /content/drive/MyDrive/Database_Skripsi_Dhio
Raw dataset: /content/drive/MyDrive/Database_Skripsi_Dhio/Dataset_Citra
Split out   : /content/drive/MyDrive/Database_Skripsi_Dhio/Dataset/Random


## 4) Script Split 60/40 secara random


In [ ]:
%%writefile split_dataset.py
import os, shutil, random, glob, sys

def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def main():
    if len(sys.argv) < 3:
        print("Usage: python split_dataset.py <SRC_RAW> <DST_SPLIT>")
        sys.exit(1)

    SRC = sys.argv[1]
    DST = sys.argv[2]
    SPLIT = (0.60, 0.40)   # train, test
    SEED = 42
    random.seed(SEED)

    print("Source:", SRC)
    print("Dest  :", DST)

    classes = [d for d in sorted(os.listdir(SRC))
               if os.path.isdir(os.path.join(SRC,d))]

    if not classes:
        print("[ERROR] Tidak ditemukan subfolder kelas di SRC.")
        sys.exit(1)

    for cls in classes:
        src_cls = os.path.join(SRC, cls)

        imgs = sorted([p for ext in ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG")
                       for p in glob.glob(os.path.join(src_cls, ext))])

        random.shuffle(imgs)
        n = len(imgs)

        if n == 0:
            print(f"[WARN] Kelas '{cls}' kosong, lewati.")
            continue

        n_train = int(SPLIT[0] * n)
        n_test  = n - n_train

        splits = {
            "train": imgs[:n_train],
            "test" : imgs[n_train:]
        }

        for split, paths in splits.items():
            dst_dir = os.path.join(DST, split, cls)
            ensure_dir(dst_dir)

            for p in paths:
                shutil.copy2(p, os.path.join(dst_dir, os.path.basename(p)))

        print(f"[OK] {cls}: {n} -> train {n_train} | test {n_test}")

    print("Done split 60/40")

if __name__ == "__main__":
    main()

Overwriting split_dataset.py


In [ ]:
if DO_SPLIT:
    !python split_dataset.py "$DATASET_RAW" "$DATASET"
else:
    print("Lewati split. Pastikan DATASET sudah berisi train/val/test.")

Source: /content/drive/MyDrive/Database_Skripsi_Dhio/Dataset_Citra
Dest  : /content/drive/MyDrive/Database_Skripsi_Dhio/Dataset/Random
[OK] Agathis borneensis - Damar Putih: 100 -> train 60 | test 40
[OK] Aquailaria beccariana - Gaharu: 100 -> train 60 | test 40
[OK] Cotylelobum Melanoxylum - Giam Tembaga: 100 -> train 60 | test 40
[OK] Diospyros Borneensis - Kayu Hitam: 100 -> train 60 | test 40
[OK] Dipterocarpus Confertus - Keruing Tempudau: 100 -> train 60 | test 40
[OK] Dipterocarpus tempehes - Keruing Asam: 100 -> train 60 | test 40
[OK] Durio dulcis - Lahung: 100 -> train 60 | test 40
[OK] Durio lanceolatus - Durian Anggang: 100 -> train 60 | test 40
[OK] Durio oxleyanus - kerantungan: 100 -> train 60 | test 40
[OK] Etlingera Balikpapanensis - Jahe Balikpapan: 100 -> train 60 | test 40
[OK] Eusideroxylon zwageri - Ulin: 100 -> train 60 | test 40
[OK] Hopea Mengerawan - damar cermin: 100 -> train 60 | test 40
[OK] Hopea Rudiformis - Merawan: 100 -> train 60 | test 40
[OK] Hopea d

## 5) Data Pipeline — RandAugment hanya TRAIN

In [ ]:
%%writefile data.py
import tensorflow as tf
from tensorflow import keras
from keras import layers

AUTOTUNE  = tf.data.AUTOTUNE
IMG_SIZE  = (480, 480)
LOAD_SIZE = (512, 512)
BATCH     = 32
SEED      = 42

def _build_aug():
    return keras.Sequential([
        layers.RandomCrop(IMG_SIZE[0], IMG_SIZE[1]),
        layers.RandomFlip("horizontal"),
        layers.RandAugment(value_range=(0, 255)),
    ])

def _with_map_train(ds, pipeline):
    return ds.map(
        lambda x, y: (pipeline(x, training=True), y),
        num_parallel_calls=AUTOTUNE
    )

def _with_map_test(ds):
    return ds.map(
        lambda x, y: (tf.cast(x, tf.float32), y),
        num_parallel_calls=AUTOTUNE
    )

def build_datasets(root="DATASET"):

    train_raw = keras.preprocessing.image_dataset_from_directory(
        directory=f"{root}/train",
        image_size=LOAD_SIZE,
        batch_size=BATCH,
        label_mode="categorical",
        shuffle=True,
        seed=SEED
    )

    test_raw = keras.preprocessing.image_dataset_from_directory(
        directory=f"{root}/test",
        image_size=IMG_SIZE,
        batch_size=BATCH,
        label_mode="categorical",
        shuffle=False,
        seed=SEED
    )

    class_names = train_raw.class_names
    num_classes = len(class_names)

    aug = _build_aug()

    train_ds = _with_map_train(train_raw, aug).prefetch(AUTOTUNE)
    test_ds  = _with_map_test(test_raw).prefetch(AUTOTUNE)

    total_train = sum(1 for _ in train_raw) * BATCH
    total_test  = sum(1 for _ in test_raw) * BATCH

    return train_ds, test_ds, num_classes, class_names, total_train, total_test

Overwriting data.py


## 6) Model EfficientNetV2 (S/M/L)

In [ ]:
%%writefile model.py
from tensorflow import keras
from keras import layers, Model, Input
from keras.applications import EfficientNetV2S, EfficientNetV2M, EfficientNetV2L

def build_model(variant: str, num_classes: int):

    if variant.upper() == "S":
        input_shape = (224, 224, 3)
        base = EfficientNetV2S(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape,
            include_preprocessing=True
        )
    elif variant.upper() == "M":
        input_shape = (480, 480, 3)
        base = EfficientNetV2M(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape,
            include_preprocessing=True
        )
    elif variant.upper() == "L":
        input_shape = (480, 480, 3)
        base = EfficientNetV2L(
            include_top=False,
            weights="imagenet",
            input_shape=input_shape,
            include_preprocessing=True
        )
    else:
        raise ValueError("variant harus 'S' | 'M' | 'L'")

    base.trainable = False

    inp = Input(shape=input_shape)
    x   = base(inp, training=False)
    x   = layers.GlobalAveragePooling2D()(x)
    x   = layers.Dense(512, activation="relu")(x)
    x   = layers.Dropout(0.3)(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = Model(inp, out)
    return model, base

Overwriting model.py


## 7) Training + F1 Macro + CSV Logger & Training Time

In [ ]:
%%writefile train.py
import time, json
import pandas as pd
import numpy as np
from tensorflow import keras
import tensorflow as tf


def train(model, base, train_ds, epochs_phase1=30, epochs_phase2=0, out_prefix="effv2l"):

    print("\n=== PHASE 1: Training HEAD saja (full backbone frozen) ===")
    print("Backbone trainable:", base.trainable)
    print(f"Total trainable params: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall")
        ]
    )

    cb_phase1 = [
        keras.callbacks.CSVLogger(f"{out_prefix}_phase1_history.csv"),
        keras.callbacks.ModelCheckpoint(
            f"{out_prefix}_best_phase1.keras",
            monitor="accuracy",
            mode="max",
            save_best_only=True,
            verbose=1
        )
    ]

    t0 = time.time()
    hist1 = model.fit(
        train_ds,
        epochs=epochs_phase1,
        callbacks=cb_phase1,
        verbose=2
    )
    t_phase1 = time.time() - t0
    print(f"\nPhase 1 selesai dalam {t_phase1/60:.1f} menit")
    print(f"Best accuracy: {max(hist1.history['accuracy']):.4f}")

    with open(f"{out_prefix}_train_time.json", "w") as f:
        json.dump({
            "phase1_seconds": float(t_phase1),
            "total_seconds" : float(t_phase1),
            "best_accuracy" : float(max(hist1.history["accuracy"]))
        }, f, indent=2)

    return model

Overwriting train.py


## 8) Evaluasi Test + Confusion Matrix

In [ ]:
%%writefile eval.py
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)

def evaluate_on_test(model, test_ds, class_names, out_prefix="effv2s"):
    """
    Evaluasi model pada test_ds.
    - y_true dikumpulkan dari dataset
    - prediksi pakai model.predict(test_ds) SATU KALI — lebih cepat & stabil
      dibanding loop per-batch yang rawan timeout/KeyboardInterrupt
    """
    # Buat folder berdasarkan out_prefix (nama variasi)
    out_dir = os.path.join("hasil", out_prefix)
    os.makedirs(out_dir, exist_ok=True)

    # Kumpulkan ground truth
    y_true = []
    for _, y in test_ds:
        y_true.extend(y.numpy().argmax(axis=1).tolist())
    y_true = np.array(y_true)

    # Predict seluruh dataset sekaligus
    print("Predicting test set...")
    y_prob = model.predict(test_ds, verbose=1)
    y_pred = y_prob.argmax(axis=1)

    # Metrics
    acc  = (y_true == y_pred).mean()
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score   (y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score       (y_true, y_pred, average="macro", zero_division=0)

    # Classification report
    report = classification_report(y_true, y_pred,
                                   target_names=class_names, zero_division=0)
    report_path = os.path.join(out_dir, f"{out_prefix}_test_report.txt")
    with open(report_path, "w") as f:
        f.write(report)
    print(f"Saved: {report_path}")

    # Confusion matrix — simpan CSV
    cm = confusion_matrix(y_true, y_pred)
    cm_csv_path = os.path.join(out_dir, f"{out_prefix}_confusion_matrix.csv")
    pd.DataFrame(cm, index=class_names, columns=class_names).to_csv(cm_csv_path)
    print(f"Saved: {cm_csv_path}")

    # Confusion matrix — plot
    cm_png_path = os.path.join(out_dir, f"{out_prefix}_confusion_matrix.png")
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest")
    plt.title("Confusion Matrix")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha="right")
    plt.yticks(tick_marks, class_names)
    plt.tight_layout()
    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.savefig(cm_png_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved: {cm_png_path}")

    print(f"TEST → Acc: {acc:.4f} | Prec(macro): {prec:.4f} | "
          f"Rec(macro): {rec:.4f} | F1(macro): {f1:.4f}")

    return {
        "accuracy":         float(acc),
        "precision_macro":  float(prec),
        "recall_macro":     float(rec),
        "f1_macro":         float(f1),
    }

Overwriting eval.py


## 9) Uji Inference Time

In [ ]:
%%writefile infer_time.py
import time, csv, json
import numpy as np
import tensorflow as tf
from data import build_datasets

def measure_inference_time(model, test_ds, n_images=100, n_runs=10, out_prefix="effv2s"):
    all_avg_times = []

    # warm-up 1 batch
    for x, _ in test_ds.take(1):
        _ = model.predict(x, verbose=0)

    with open(f"{out_prefix}_inference_times.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["idx", "mean_time_sec", "std_time_sec"])

        grabbed = 0
        for x, _ in test_ds.unbatch().batch(1):
            if grabbed >= n_images:
                break
            times_per_image = []
            for _ in range(n_runs):
                t0 = time.time()
                model.predict(x, verbose=0)
                dt = time.time() - t0
                times_per_image.append(dt)

            mean_t = float(np.mean(times_per_image))
            std_t  = float(np.std(times_per_image))
            all_avg_times.append(mean_t)
            w.writerow([grabbed, f"{mean_t:.6f}", f"{std_t:.6f}"])
            grabbed += 1

    overall_mean = float(np.mean(all_avg_times))
    overall_std  = float(np.std(all_avg_times))

    print(f"Inference time over {grabbed} images x {n_runs} runs:")
    print(f"  Mean : {overall_mean:.6f} s")
    print(f"  Std  : {overall_std:.6f} s")

    return overall_mean, overall_std


if __name__ == "__main__":
    VARIANT = "L"  # ganti S / M / L
    OUT     = f"effv2{VARIANT.lower()}"
    ROOT_DATASET = "/content/drive/MyDrive/Database_Skripsi_Dhio/Dataset/Random"
    HASIL_DIR    = f"/content/drive/MyDrive/Database_Skripsi_Dhio/hasil/{OUT}"

    model = tf.keras.models.load_model(f"{HASIL_DIR}/{OUT}_best_phase1.keras")
    print("Model loaded:", model.name)

    _, test_ds, _, _, _, _ = build_datasets(root=ROOT_DATASET)

    mean_t, std_t = measure_inference_time(model, test_ds, n_images=100, n_runs=10, out_prefix=f"{HASIL_DIR}/{OUT}")

    with open(f"{HASIL_DIR}/{OUT}_inference_avg.json", "w") as f:
        json.dump({"mean_inference_time_sec": mean_t, "std_inference_time_sec": std_t}, f, indent=2)

    print(f"Selesai. Hasil disimpan di: {HASIL_DIR}")

Overwriting infer_time.py


In [ ]:
!python infer_time.py

2026-04-13 13:10:04.603955: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776085804.625283    8227 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776085804.632431    8227 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776085804.651648    8227 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776085804.651696    8227 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776085804.651701    8227 computation_placer.cc:177] computation placer alr

## 10) Main (Skenario eksperimen S/M/L)

In [ ]:
%%writefile main.py
import json, os
import tensorflow as tf
  from data import build_datasets
  from model import build_model
from train import train
from eval import evaluate_on_test
from infer_time import measure_inference_time

# === PARAMETER SKENARIO ===

VARIANT       = "L"
EPOCHS_PHASE1 = 50
EPOCHS_PHASE2 = 0
OUT           = f"effv2{VARIANT.lower()}"
ROOT_DATASET  = "/content/drive/MyDrive/Database_Skripsi_Dhio/Dataset/Random"

tf.random.set_seed(42)

# === Data ===
train_ds, test_ds, num_classes, class_names, total_train, total_test = build_datasets(root=ROOT_DATASET)
class_names = tuple(class_names)
print(f"Classes ({num_classes}):", class_names)
print(f"Total train samples : ~{total_train}")
print(f"Total test samples  : ~{total_test}")
print(f"Augmentasi          : Online (RandAugment) — jumlah data tidak berubah")

# === Model ===
model, base = build_model(VARIANT, num_classes)
model.summary()

# === Train (Phase 1 saja) ===
model = train(
    model, base, train_ds,
    epochs_phase1=EPOCHS_PHASE1,
    epochs_phase2=EPOCHS_PHASE2,
    out_prefix=OUT
)

# === Evaluate (TEST) ===
metrics_test = evaluate_on_test(model, test_ds, class_names, out_prefix=OUT)
with open(f"{OUT}_test_metrics.json", "w") as f:
        json.dump(metrics_test, f, indent=2)

# === Inference time ===
avg_inf = measure_inference_time(model, test_ds, n_images=100, out_prefix=OUT)
with open(f"{OUT}_inference_avg.json", "w") as f:
    json.dump({"avg_inference_time_sec": avg_inf}, f, indent=2)

print("\nSelesai. File output tersimpan dengan prefix:", OUT)

Overwriting main.py


## 11) Jalankan skenario (ubah VARIANT di `main.py` bila perlu)

In [ ]:
!python main.py

2026-04-08 19:26:46.300492: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775676406.319753    2639 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775676406.326126    2639 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775676406.341258    2639 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775676406.341284    2639 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775676406.341287    2639 computation_placer.cc:177] computation placer alr

In [ ]:
!ls

data.py  Dataset_Citra	 eval.py	main.py   __pycache__	    train.py
Dataset  Dataset_Mentah  infer_time.py	model.py  split_dataset.py
